In [ ]:
!pip install -q "gensim>=4.3.3"
# numpy 互換エラーが出たら: ランタイム > セッションを再起動 して再実行


In [ ]:
# ===== knock79: アーキテクチャの変更 — Transformer エンコーダ分類器(GPU) =====
# 78 までの BoW(平均)を Transformer エンコーダに置き換える(自由設計課題)。
# 新部品: 位置エンコーディング / self-attention + パディングマスク / TransformerEncoder / Adam。
#
# ★76 との重要な違い: エンコーダ出力は PAD 位置でも非ゼロになる(attention が値を作る)。
#   だからプーリングの前に「必ずマスクを掛けてから」和を取る。76 は padding_idx で
#   埋め込みが PAD=0 だったので和がそのまま安全だったが、ここではそうならない。
#
# 期待値: スクラッチ学習なので BoW(0.77)を超えないことが多い。狙いは実装練習と GPU 体感。
# 実行前に: ランタイム > ランタイムのタイプを変更 > GPU

import csv
import math
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import gensim.downloader as api

PAD = "<PAD>"
LIMIT = 100000
BATCH_SIZE = 64
LR = 1e-3  # Transformer は Adam + 小さめ lr
EPOCHS = 10
NHEAD = 6  # d_model(300)を割り切る必要: 300/6=50
NUM_LAYERS = 2
DIM_FF = 512
DROPOUT = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 70: 埋め込み行列 ----
kv = api.load("word2vec-google-news-300")
vocab = kv.index_to_key[:LIMIT]
demb = kv.vector_size  # 300 = d_model
E = np.zeros((len(vocab) + 1, demb), dtype=np.float32)
E[1:] = kv.vectors[:LIMIT]
word2id = {PAD: 0}
for i, w in enumerate(vocab):
    word2id[w] = i + 1


# ---- 71: データ ----
def load_sst2(path):
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        return [(s, l) for s, l in reader]


def text_to_ids(text, word2id):
    return [word2id[w] for w in text.split() if w in word2id]


def build_dataset(data, word2id):
    ds = []
    for s, l in data:
        ids = text_to_ids(s, word2id)
        if not ids:
            continue
        ds.append(
            {
                "text": s,
                "label": torch.tensor([float(l)]),
                "input_ids": torch.tensor(ids, dtype=torch.long),
            }
        )
    return ds


# ---- 75: collate ----
def collate(batch):
    batch = sorted(batch, key=lambda ex: ex["input_ids"].size(0), reverse=True)
    seqs = [ex["input_ids"] for ex in batch]
    labels = [ex["label"] for ex in batch]
    input_ids = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return {"input_ids": input_ids, "label": torch.stack(labels)}


# ---- 位置エンコーディング(sinusoidal, 学習パラメータ無し) ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()  # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)  # 偶数次元
        pe[:, 1::2] = torch.cos(pos * div)  # 奇数次元
        # 学習しないので buffer(model.to(device) で一緒に移動する)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):  # x: (B, L, d_model)
        return x + self.pe[:, : x.size(1)]  # 先頭 L 個の位置符号を足す


# ---- 79: Transformer 分類器 ----
class TransformerClassifier(nn.Module):
    def __init__(self, E):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(E), freeze=True, padding_idx=0
        )
        d_model = self.emb.embedding_dim  # 300
        self.pos = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=NHEAD,
            dim_feedforward=DIM_FF,
            dropout=DROPOUT,
            batch_first=True,  # 入力を (B, L, d) にする
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=NUM_LAYERS)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, input_ids):
        pad_mask = input_ids == 0  # (B, L) True=PAD(attention で無視される位置)
        x = self.emb(input_ids)  # (B, L, 300)
        x = self.pos(x)  # 位置情報を加える
        x = self.encoder(x, src_key_padding_mask=pad_mask)  # (B, L, 300) 文脈化
        # ★masked-mean プーリング: encoder 出力は PAD でも非ゼロ → 掛けてから和を取る。
        keep = (~pad_mask).unsqueeze(-1).float()  # (B, L, 1) 実トークン=1
        summed = (x * keep).sum(dim=1)  # (B, 300)
        lengths = keep.sum(dim=1)  # (B, 1) 実トークン数
        feat = summed / lengths  # (B, 300)
        return torch.sigmoid(self.fc(feat))  # (B, 1)


def train_model(model, train_data):
    loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate
    )
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)  # ★ SGD ではなく Adam
    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in loader:
            ids = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            prob = model(ids)
            loss = criterion(prob, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * y.size(0)
        print(f"epoch {epoch}: loss = {total / len(train_data):.4f}")
    return model


def accuracy(model, data):
    loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            pred = (model(ids) >= 0.5).float()
            correct += (pred == y).sum().item()
    return correct / len(data)


# ---- 実行 ----
from google.colab import files  # noqa: E402

print("train.tsv と dev.tsv を選択してアップロード:")
files.upload()

train = build_dataset(load_sst2("train.tsv"), word2id)
dev = build_dataset(load_sst2("dev.tsv"), word2id)
print("train:", len(train), "dev:", len(dev))

model = TransformerClassifier(E).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("学習パラメータ数:", n_params)

t0 = time.time()
train_model(model, train)
print("train time:", round(time.time() - t0, 1), "s")
print("dev accuracy:", accuracy(model, dev))
